<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Turistas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_turistas.csv"

df = pd.read_csv(url)

df.head()

,id_visita,fecha,pais_origen,via_ingreso,motivo_viaje,departamento_visitado,noches_estadia,gasto_total
0,1,2024-02-01,Canada,Terrestre,Negocios,Santa Ana,8,2233
1,2,2024-12-30,Estados Unidos,Terrestre,Familiares,Sonsonate,5,408
2,3,2023-05-11,Guatemala,Aerea,Familiares,Sonsonate,7,572
3,4,2024-07-18,Canada,Aerea,Ocio,Sonsonate,2,241
4,5,2024-02-05,Estados Unidos,Terrestre,Negocios,San Salvador,10,847


Validación de estructura

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_visita              20000 non-null  int64 
 1   fecha                  20000 non-null  object
 2   pais_origen            20000 non-null  object
 3   via_ingreso            20000 non-null  object
 4   motivo_viaje           20000 non-null  object
 5   departamento_visitado  20000 non-null  object
 6   noches_estadia         20000 non-null  int64 
 7   gasto_total            20000 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 1.2+ MB


Mostrar dimensiones

In [3]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 20000
Columnas: 8


Mostrar estadísticas

In [4]:
df.describe()

,id_visita,noches_estadia,gasto_total
count,20000.000000,20000.000000,20000.000000
mean,10000.500000,5.990350,1313.844150
std,5773.647028,3.176187,688.224707
min,1.000000,1.000000,120.000000
25%,5000.750000,3.000000,717.000000
50%,10000.500000,6.000000,1312.000000
75%,15000.250000,9.000000,1914.000000
max,20000.000000,11.000000,2499.000000


Verificación de valores nulos

In [5]:
df.isnull().sum()

,0
id_visita,0
fecha,0
pais_origen,0
via_ingreso,0
motivo_viaje,0
departamento_visitado,0
noches_estadia,0
gasto_total,0


Eliminación de duplicados

In [6]:
df.drop_duplicates(inplace=True)

Confirmar registros

In [7]:
df.shape

(20000, 8)

Conversión de fechas

In [8]:
df['fecha'] = pd.to_datetime(df['fecha'])

Crear atributos de tiempo

In [9]:
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["dia"] = df["fecha"].dt.day
df["trimestre"] = df["fecha"].dt.quarter

Crear Dimensión Tiempo

In [10]:
dim_tiempo = df[[
    "fecha",
    "anio",
    "mes",
    "dia",
    "trimestre"
]].drop_duplicates()

dim_tiempo = dim_tiempo.reset_index(drop=True)

dim_tiempo["idTiempo"] = dim_tiempo.index + 1

dim_tiempo.head()

,fecha,anio,mes,dia,trimestre,idTiempo
0,2024-02-01,2024,2,1,1,1
1,2024-12-30,2024,12,30,4,2
2,2023-05-11,2023,5,11,2,3
3,2024-07-18,2024,7,18,3,4
4,2024-02-05,2024,2,5,1,5


Crear Dimensión País

In [11]:
dim_pais = df[[
    "pais_origen"
]].drop_duplicates()

dim_pais = dim_pais.reset_index(drop=True)

dim_pais["idPais"] = dim_pais.index + 1

dim_pais.rename(columns={
    "pais_origen":"pais"
}, inplace=True)

dim_pais.head()

,pais,idPais
0,Canada,1
1,Estados Unidos,2
2,Guatemala,3
3,Mexico,4
4,Honduras,5


Crear Dimensión Departamento

In [12]:
dim_departamento = df[[
    "departamento_visitado"
]].drop_duplicates()

dim_departamento = dim_departamento.reset_index(drop=True)

dim_departamento["idDepartamento"] = dim_departamento.index + 1

dim_departamento.rename(columns={
    "departamento_visitado":"nombre"
}, inplace=True)

dim_departamento.head()

,nombre,idDepartamento
0,Santa Ana,1
1,Sonsonate,2
2,San Salvador,3
3,San Miguel,4
4,La Libertad,5


Crear Dimensión Motivo

In [13]:
dim_motivo = df[[
    "motivo_viaje"
]].drop_duplicates()

dim_motivo = dim_motivo.reset_index(drop=True)

dim_motivo["idMotivo"] = dim_motivo.index + 1

dim_motivo.rename(columns={
    "motivo_viaje":"motivo"
}, inplace=True)

dim_motivo.head()

,motivo,idMotivo
0,Negocios,1
1,Familiares,2
2,Ocio,3
3,Salud,4
4,Religioso,5


Crear Dimensión Vía de Ingreso

In [14]:
dim_via = df[[
    "via_ingreso"
]].drop_duplicates()

dim_via = dim_via.reset_index(drop=True)

dim_via["idVia"] = dim_via.index + 1

dim_via.rename(columns={
    "via_ingreso":"tipoVia"
}, inplace=True)

dim_via.head()

,tipoVia,idVia
0,Terrestre,1
1,Aerea,2


Construir FACT_TURISTAS

In [15]:
fact_turistas = df.copy()

fact_turistas = fact_turistas.merge(
    dim_pais,
    left_on="pais_origen",
    right_on="pais"
)

fact_turistas = fact_turistas.merge(
    dim_departamento,
    left_on="departamento_visitado",
    right_on="nombre"
)

fact_turistas = fact_turistas.merge(
    dim_motivo,
    left_on="motivo_viaje",
    right_on="motivo"
)

fact_turistas = fact_turistas.merge(
    dim_via,
    left_on="via_ingreso",
    right_on="tipoVia"
)

fact_turistas = fact_turistas.merge(
    dim_tiempo,
    on="fecha"
)

Seleccionar columnas finales

In [16]:
fact_turistas = fact_turistas[[
    "id_visita",
    "idPais",
    "idDepartamento",
    "idTiempo",
    "idMotivo",
    "idVia",
    "noches_estadia",
    "gasto_total"
]]

Exportar resultados

In [17]:
dim_tiempo.to_csv("dim_tiempo.csv", index=False)
dim_pais.to_csv("dim_pais.csv", index=False)
dim_departamento.to_csv("dim_departamento.csv", index=False)
dim_motivo.to_csv("dim_motivo.csv", index=False)
dim_via.to_csv("dim_via.csv", index=False)

fact_turistas.to_csv("fact_turistas_dw.csv", index=False)